# 🍫  Chocolate Sales Analytics | Advanced Pandas + Plotly | Business Insights
### *Executive Performance Handbook & Analytical Framework*

---

## 🚀 Business Impact

- This analysis uncovers key revenue drivers, identifies high-performing products, and highlights growth opportunities across markets.

## 📝 Description
___

This project analyzes chocolate sales data to uncover insights on:
- Sales performance
- Product demand
- Growth trends
- Salesperson efficiency

Tools used:
- Pandas
- Plotly

**End-to-end data analysis of chocolate sales dataset using Pandas and Plotly.**

Includes:
- Revenue analysis
- Growth trends
- Efficiency metrics
- Business insights

**Built as a portfolio project demonstrating real-world analytics skills.**

## 🎯 Project Objective
- This handbook serves as a comprehensive business intelligence audit of global chocolate sales. The goal is to transition from raw descriptive statistics to **prescriptive insights**, identifying high-value sales channels, optimizing product distribution, and uncovering growth opportunities using FAANG-style analytical rigor.

---

In [1]:
# This Python 3 environment comes with many helpful analytics libraries installed
# It is defined by the kaggle/python Docker image: https://github.com/kaggle/docker-python
# For example, here's several helpful packages to load

import numpy as np # linear algebra
import pandas as pd # data processing, CSV file I/O (e.g. pd.read_csv)

# Input data files are available in the read-only "../input/" directory
# For example, running this (by clicking run or pressing Shift+Enter) will list all files under the input directory

import os
for dirname, _, filenames in os.walk('/kaggle/input'):
    for filename in filenames:
        print(os.path.join(dirname, filename))

# You can write up to 20GB to the current directory (/kaggle/working/) that gets preserved as output when you create a version using "Save & Run All" 
# You can also write temporary files to /kaggle/temp/, but they won't be saved outside of the current session

/kaggle/input/datasets/saidaminsaidaxmadov/chocolate-sales/ChocolateSales.pbix
/kaggle/input/datasets/saidaminsaidaxmadov/chocolate-sales/Chocolate Sales (2).csv
/kaggle/input/datasets/saidaminsaidaxmadov/chocolate-sales/ChocolateSales.pdf
/kaggle/input/datasets/saidaminsaidaxmadov/chocolate-sales/ChocolateSales.pbit


In [2]:
df=pd.read_csv('/kaggle/input/datasets/saidaminsaidaxmadov/chocolate-sales/Chocolate Sales (2).csv')

### 🧹 Data Clean and Handling Missing Values
___

In [3]:
# Change Data Type of Data From Object to Datatime
df['Date']=pd.to_datetime(df['Date'], format='mixed')
df['Year']=df['Date'].dt.year
df['Month']=df['Date'].dt.month


In [4]:
# Change Data Types Of Amount from string to Numeric
df['Amount']=df['Amount'].str.replace('$',"")
df['Amount']=df['Amount'].str.replace(',',"")
df['Amount']=pd.to_numeric(df['Amount'])

## 📑 Data Dictionary
Understanding the grain of the data is the first step in any high-level analysis.

| Column | Data Type | Description |
| :--- | :--- | :--- |
| **Sales Person** | `String` | Unique identifier for the individual responsible for the account. |
| **Country** | `String` | The geographic market where the transaction occurred. |
| **Product** | `String` | The specific SKU (Stock Keeping Unit) sold. |
| **Date** | `Datetime64` | The timestamp of the transaction. |
| **Amount** | `Float64` | Total revenue generated from the sale ($). |
| **Boxes Shipped**| `Integer` | Total volume of physical units delivered. |

---

## 🛠️ Phase 1: Descriptive Analytics (The Vital Signs)
*Focus: Establishing a baseline for business health.*

#####  Q1: Top Performing Salespersons

- Find top 5 salespersons by total revenue.

In [5]:
import numpy as np
import pandas as pd
import plotly.express as px
import plotly.graph_objects as go
result=(
    df.groupby('Sales Person')['Amount']
    .sum()
    .sort_values(ascending=False)
    .nlargest(5)
    .reset_index()  
)
result
fig=px.bar(result,x='Sales Person',y='Amount',template='plotly_dark',color='Amount',color_continuous_scale=px.colors.sequential.Viridis, log_y=True)
fig.show()

##### 🔹 Q2: Country-wise Revenue

- Which countries generate the highest total revenue?


In [6]:
result = (
    df
    .groupby('Country', as_index=False)['Amount']
    .sum()
    .sort_values(by='Amount', ascending=False)
    .nlargest(5, 'Amount')
)

fig = px.bar(
    result,
    x='Amount',
    y='Country',
    orientation='h',
    color='Amount',
    color_continuous_scale=px.colors.diverging.RdBu,
    log_x=True,
    title='Top 5 Countries by Revenue',
    template='plotly_dark',
    text='Amount'
)

fig.show()

##### 🔹 Q3: Product Performance

- Which products generate the most revenue?

In [7]:
df.columns

Index(['Sales Person', 'Country', 'Product', 'Date', 'Amount', 'Boxes Shipped',
       'Year', 'Month'],
      dtype='object')

In [8]:
result = (
    df
    .groupby('Product', as_index=False)['Amount']
    .sum()
    .sort_values(by='Amount', ascending=False)
    .nlargest(5, 'Amount')
)

fig = px.bar(
    result,
    x='Product',
    y='Amount',
    template='plotly_dark',
    color='Amount',
    log_y=True,
    color_continuous_scale=px.colors.sequential.OrRd,
    title='Top 5 Products by Revenue',
    text='Amount'
)

fig.show()

##### 🔹 Q4: Total Sales Trend Over Time

- How does total sales vary over time (monthly/yearly)?

In [9]:
result1=(
    df
    .groupby('Year')["Amount"]
    .sum()
    .reset_index()  
)

result2=(
    df
    .groupby('Month')["Amount"]
    .sum()
    .reset_index()  
)

from plotly.subplots import make_subplots
import plotly.graph_objects as go

# Create subplot (1 row, 2 columns)
fig = make_subplots(
    rows=1, cols=2,
    subplot_titles=("Yearly Sales", "Monthly Sales")
)

# 🔹 Plot 1: Yearly
fig.add_trace(
    go.Scatter(
        x=result1['Year'],
        y=result1['Amount'],
        mode='lines+markers',
        name='Yearly Sales'
    ),
    row=1, col=1
)

# 🔹 Plot 2: Monthly
fig.add_trace(
    go.Scatter(
        x=result2['Month'],
        y=result2['Amount'],
        mode='lines+markers',
        name='Monthly Sales'
    ),
    row=1, col=2
)

# Layout
fig.update_layout(
    title="Sales Analysis (Year vs Month)",
    template='plotly_dark'
)

fig.show()

##### 🔹 Q5: Average Order Value

- What is the average revenue per transaction?

In [10]:
avg_revenue = df['Amount'].mean()
print(avg_revenue)

6030.338775137113


# 🧠 SECTION 2: Intermediate Analysis
___

##### 🔹 Q6: Salesperson Efficiency

- Which salesperson generates highest revenue per box shipped?

In [11]:
result = (
    df
    .groupby('Sales Person', as_index=False)
    .agg(
        Total_Revenue=('Amount', 'sum'),
        Total_Box=('Boxes Shipped', 'sum')
    )
    .assign(
        Salesperson_Efficiency=lambda x: x['Total_Revenue'] / x['Total_Box']
    )
    .sort_values(by='Salesperson_Efficiency',ascending=False)
    .nlargest(5, 'Salesperson_Efficiency')
    .reset_index()
)

fig = px.pie(
    result,
    names='Sales Person',
    values='Salesperson_Efficiency',
    color='Salesperson_Efficiency',
    color_discrete_sequence=px.colors.diverging.oxy,
    title='Top 5 Salespersons by Efficiency'
)

fig.show()

##### 🔹 Q7: Product Demand Distribution

- How are box shipments distributed across products?

In [12]:
# 1. Prepare the data
product_demand = (
    df
    .groupby('Product', as_index=False)
    .agg(
        Total_boxes=('Boxes Shipped', 'sum') 
    )
    .nlargest(7, 'Total_boxes')
    .reset_index(drop=True)
)

# 2. Plot the data
fig = px.bar(
    product_demand,            
    x='Total_boxes',           
    y='Product',
    log_x=True,
    template='plotly_dark',
    color='Product',
    color_discrete_sequence=px.colors.sequential.PuBu, 
    title='Top Products In Demand'         
)

fig.show()

##### 🔹 Q8: Country-wise Product Popularity

- Which product is most popular in each country?

In [13]:
pro_pop=(
    df
    .groupby('Country')['Amount']
    .sum()
    .sort_values(ascending=False)
    .reset_index()
)

import plotly.express as px
fig = px.choropleth(
    pro_pop,
    locations='Country',           
    locationmode='country names', 
    color='Amount',                
    hover_name='Country',         
    title='Total Sales Amount by Country',
    color_continuous_scale=px.colors.sequential.Viridis, 
    template='plotly_dark'         
)

fig.update_layout(
    geo=dict(
        showframe=False,
        showcoastlines=True,
        projection_type='equirectangular'
    )
)

fig.show()


##### 🔹 Q9: Monthly Growth Rate

- Calculate month-over-month growth in sales.

In [14]:
monthly_sales = (
    df
    .groupby('Month')['Amount']
    .sum()
    .reset_index()
)

monthly_sales['Prev_Sales'] = monthly_sales['Amount'].shift(1)

monthly_sales['MoM_%'] = (
    (monthly_sales['Amount'] - monthly_sales['Prev_Sales'])
    / monthly_sales['Prev_Sales'] * 100
).round(2)


monthly_sales = monthly_sales.fillna(0)
monthly_sales = monthly_sales.sort_values('Month').reset_index(drop=True)


cumulative_data = []
months = monthly_sales['Month'].unique()

for m in months:
    subset = monthly_sales[monthly_sales['Month'] <= m].copy()
    subset['Frame'] = m 
    cumulative_data.append(subset)

animated_df = pd.concat(cumulative_data, ignore_index=True)

# 3. Create the Animated Plot
fig = px.line(
    animated_df,
    x='Month',
    y='MoM_%', 
    line_shape='spline',
    template='plotly_dark',
    log_y=False,          
    title='Month-over-Month % Change (Animated)',
    animation_frame='Frame' 
)

# 4. Fix Axis Ranges
x_min = animated_df['Month'].min()
x_max = animated_df['Month'].max()
y_min = animated_df['MoM_%'].min()
y_max = animated_df['MoM_%'].max()

# Adjusting for a little padding so the line isn't touching the edge
fig.update_layout(
    xaxis=dict(range=[x_min, x_max], autorange=False),
    yaxis=dict(range=[y_min - 5, y_max + 5], autorange=False)
)

fig.layout.updatemenus[0].buttons[0].args[1]["frame"]["duration"] = 500

fig.show()

##### 🔹 Q10: High-Value Transactions

- Find transactions where sales amount > 90th percentile.


In [15]:
high_value_transactions= df[df['Amount'] > df['Amount'].quantile(0.90)]
high_value_transactions

fig = px.box(df, 
             y="Amount", 
             points="outliers", 
             template="plotly_dark",
             title="Transaction Value Distribution (Identifying High-Value Outliers)")

# Add a horizontal line at the 90th percentile
cutoff = df['Amount'].quantile(0.90)
fig.add_hline(y=cutoff, line_dash="dash", line_color="orange", 
              annotation_text=f"90th Percentile: {cutoff:.2f}")

fig.show()

# 🔥 SECTION 3: Advanced Business Insights
---

##### 🔹 Q11: Sales Concentration

- What % of total revenue is generated by top 20% salespersons?


In [16]:
top_sales = (
    df
    .groupby('Sales Person')['Amount']
    .sum()
    .sort_values(ascending=False)
)

top_20 = top_sales.head(int(len(top_sales) * 0.2))
pct = ((top_20.sum() / top_sales.sum()) * 100).round(2)
print(pct)


# Step A: Get Total Sales per Salesperson (as a DataFrame)
top_sales_df = (
    df
    .groupby('Sales Person')['Amount']
    .sum()
    .sort_values(ascending=False)
    .reset_index()
)

total_revenue = top_sales_df['Amount'].sum()

# Step B: Calculate Cumulative Sum and Cumulative Percentage
top_sales_df['cumulative_sum'] = top_sales_df['Amount'].cumsum()
top_sales_df['cumulative_perc'] = (top_sales_df['cumulative_sum'] / total_revenue) * 100

# Optional: Limit to top X salespeople for readability if the list is huge
# top_sales_df = top_sales_df.head(20) # Uncomment if needed


# 2. Creating the Dual-Axis Pareto Chart

# Create figure with secondary y-axis
fig = make_subplots(specs=[[{"secondary_y": True}]])

# A. Add the Bar Trace (Sales Amount - Primary Y-Axis)
fig.add_trace(
    go.Bar(
        x=top_sales_df['Sales Person'],
        y=top_sales_df['Amount'],
        name="Sales Amount",
        marker_color='#5A9BD5', # Cool Blue
        opacity=0.8
    ),
    secondary_y=False, # This trace uses the primary (left) y-axis
)

# B. Add the Line Trace (Cumulative % - Secondary Y-Axis)
fig.add_trace(
    go.Scatter(
        x=top_sales_df['Sales Person'],
        y=top_sales_df['cumulative_perc'],
        name="Cumulative %",
        mode='lines+markers',
        line=dict(color='#ED7D31', width=3), # Orange
        marker=dict(size=8)
    ),
    secondary_y=True, # This trace uses the secondary (right) y-axis
)


# 3. Styling and Annotations

# Identify the cutoff point for your original calculation (top 20%)
top_20_cutoff_index = int(len(top_sales_df) * 0.2)
# Ensure index isn't out of bounds if dataset is very small
top_20_cutoff_index = min(top_20_cutoff_index, len(top_sales_df) - 1) 

if top_20_cutoff_index >= 0:
    salesperson_at_cutoff = top_sales_df.iloc[top_20_cutoff_index]['Sales Person']
    percent_at_cutoff = top_sales_df.iloc[top_20_cutoff_index]['cumulative_perc']

    # Add visual indicators for the 80/20 rule application
    
    # 1. Horizontal dotted line across the chart at the percentage level
    fig.add_hline(y=percent_at_cutoff, line_dash="dash", line_color="orange", secondary_y=True)
    
    # 2. Vertical line separating top 20% salespeople
    fig.add_vline(x=top_20_cutoff_index, line_dash="dash", line_color="#444")

    # 3. Annotation showing your "pct" calculation result
    # We use annotations instead of simple print() so it's on the chart
    fig.add_annotation(
        x=top_20_cutoff_index,
        y=percent_at_cutoff,
        secondary_y=True,
        text=f"The top 20% salespeople<br>({salesperson_at_cutoff})<br>contribute {pct}%",
        showarrow=True,
        arrowhead=2,
        ax=40,
        ay=40,
        bgcolor="rgba(0,0,0,0.8)",
        bordercolor="#ED7D31"
    )

# Set final layout
fig.update_layout(
    title_text=f"Pareto Analysis: Salesperson Contribution (Top 20% contribute {pct}%)",
    template='plotly_dark',
    hovermode='x unified', # Shows both values on one hover label
    legend=dict(orientation="h", yanchor="bottom", y=1.02, xanchor="right", x=1)
)

# Set x-axis title
fig.update_xaxes(title_text="Sales Person")

# Set y-axes titles
fig.update_yaxes(title_text="<b>Primary</b> Sales Amount ($)", secondary_y=False)
fig.update_yaxes(title_text="<b>Secondary</b> Cumulative %", secondary_y=True, range=[0, 105])

fig.show()

25.52


##### 🔹 Q12: Product Profitability Proxy

- Which products generate highest revenue per box shipped?


In [17]:
high_rev_prod = (
    df
    .groupby('Product', as_index=False) # as_index=False makes plotting much easier
    .agg(
        Total_Revenue=('Amount', 'sum'),
        Total_Boxes=('Boxes Shipped', 'sum')
    )
    .assign(
        Prod_Profitability=lambda x: x['Total_Revenue'] / x['Total_Boxes']
    )
    .sort_values(by='Prod_Profitability', ascending=False)
)

high_rev_prod.head(5)

,Product,Total_Revenue,Total_Boxes,Prod_Profitability
5,Almond Choco,890454.65,20558,43.314265
21,White Choc,1054257.00,25158,41.905438
19,Smooth Sliky Salty,1120201.09,26969,41.536619
17,Peanut Butter Cubes,1036591.09,25339,40.908919
2,85% Dark Bars,955268.24,23828,40.090156


##### 🔹 Q13: Seasonal Trends

- Which months have peak sales?


In [18]:
import plotly.express as px


peak_months = (
    df
    .groupby('Month')['Amount']
    .sum()
    .nlargest(5)
    .reset_index()
)

# 2. Create the Donut Chart
fig = px.pie(
    peak_months, 
    values='Amount', 
    names='Month', 
    hole=0.5,                               
    template='plotly_dark',
    color_discrete_sequence=px.colors.sequential.Pinkyl, 
    title='Top 5 Peak Months by Revenue'
)

# Optional: Improve the styling
fig.update_traces(
    textinfo='percent+label', 
    pull=[0.1, 0.01, 0.01, 0.01, 0.01]                   # Slightly "pop out" the biggest slice
)

fig.show()

##### 🔹 Q14: Sales Consistency

- Which salespersons have most consistent performance (lowest variance)?

In [19]:
sales_consistency = (
    df
    .groupby('Sales Person', as_index=False)
    .agg(
        total_sales=('Amount', 'sum'),
        avg_sales=('Amount', 'mean'),
        variance=('Amount', 'var')
    )
    .sort_values(by='variance', ascending=True)
    .round(2)
)

sales_consistency.head(3)

,Sales Person,total_sales,avg_sales,variance
24,Wilone O'Kielt,439961.92,4313.35,11142435.03
22,Roddy Speechley,808359.58,6266.35,12985027.09
13,Jehu Rudeforth,708505.03,5492.29,13466554.23


##### 🔹 Q15: Country Growth Trends

- Which countries show fastest growth in sales over time?


In [20]:
country_month = (
    df
    .groupby(['Country', 'Month'], as_index=False)
    .agg(total_sales=('Amount', 'sum'))
    .sort_values(['Country', 'Month'])
)
country_month['growth_%'] = (
    country_month
    .groupby('Country')['total_sales']
    .pct_change() * 100
)
growth_summary = (
    country_month
    .groupby('Country', as_index=False)['growth_%']
    .mean()
    .sort_values(by='growth_%', ascending=False)
)

growth_summary.head(3)



fig = px.bar(
    growth_summary.head(3),
    x='Country',
    y='growth_%',
    color='growth_%',
    log_y=True,
    color_continuous_scale=px.colors.sequential.Viridis,
    title='Top 3 Fastest Growing Countries'
)

fig.show()

# 💀 Phase 4: FAANG-Level Advanced Modeling
*Focus: Statistical rigor and predictive capabilities.*
___

##### 🔹 Q16: Salesperson Ranking System
Rank salespersons based on:
- Total revenue
- Consistency
- Efficiency

In [21]:
sales_rank = (
    df
    .groupby('Sales Person', as_index=False)
    .agg(
        total_revenue=('Amount', 'sum'),
        avg_sales=('Amount', 'mean'),
        variance=('Amount', 'var'),
        total_boxes=('Boxes Shipped', 'sum')
    )
    .assign(
        efficiency=lambda x: x['total_revenue'] / x['total_boxes']
    )
)
sales_rank = sales_rank[sales_rank['total_boxes'] > 0]

sales_rank = sales_rank.assign(
    revenue_rank = sales_rank['total_revenue'].rank(ascending=False),
    consistency_rank = sales_rank['variance'].rank(ascending=True),   # low variance = good
    efficiency_rank = sales_rank['efficiency'].rank(ascending=False)
)
sales_rank['final_score'] = (
    sales_rank['revenue_rank'] +
    sales_rank['consistency_rank'] +
    sales_rank['efficiency_rank']
)
sales_rank = sales_rank.sort_values('final_score',ascending=False)

sales_rank[['Sales Person', 'final_score']].head(10)

,Sales Person,final_score
8,Dotty Strutley,67.0
12,Jan Morforth,62.0
6,Curtice Advani,55.0
0,Andria Kimpton,53.0
4,Camilla Castle,51.0
15,Karlen McCaffrey,48.0
18,Mallorie Waber,45.0
2,Beverie Moffet,45.0
21,Rafaelita Blaksland,44.0
11,Husein Augar,43.0


##### 🔹 Q17: Outlier Detection

- Identify abnormal transactions (very high or low sales).

In [22]:
Q1 = df['Amount'].quantile(0.25)
Q3 = df['Amount'].quantile(0.75)

IQR = Q3 - Q1

lower_bound = Q1 - 1.5 * IQR
upper_bound = Q3 + 1.5 * IQR

outliers = df[
    (df['Amount'] < lower_bound) | 
    (df['Amount'] > upper_bound)
]

outliers.head()

,Sales Person,Country,Product,Date,Amount,Boxes Shipped,Year,Month
66,Van Tuxwell,Australia,Organic Choco Syrup,2022-10-08,19453.0,14,2022,10
135,Van Tuxwell,India,Organic Choco Syrup,2022-05-16,19929.0,174,2022,5
212,Marney O'Breen,UK,Smooth Sliky Salty,2022-05-13,18991.0,88,2022,5
434,Jan Morforth,New Zealand,Mint Chip Choco,2022-06-30,18340.0,285,2022,6
543,Ches Bonnell,India,Peanut Butter Cubes,2022-01-27,22050.0,208,2022,1


#### 📈 Visualization

In [23]:
import plotly.express as px

fig = px.violin(
    df, 
    y="Amount", 
    box=True,              
    points='all',          
    hover_data=df.columns, 
    color_discrete_sequence=['#FF69B4'], 
    template="plotly_dark",
    title="Distribution of Transaction Amounts with Outliers"
)


fig.add_hline(y=upper_bound, line_dash="dash", line_color="red", 
              annotation_text="Upper Outlier Bound")
fig.add_hline(y=lower_bound, line_dash="dash", line_color="green", 
              annotation_text="Lower Outlier Bound")

fig.show()

##### 🔹 Q18: Revenue vs Volume Relationship

- Does more boxes shipped always mean higher revenue?

In [24]:
rv_data = (
    df
    .groupby('Sales Person', as_index=False)
    .agg(
        total_boxes=('Boxes Shipped', 'sum'),
        total_revenue=('Amount', 'sum')
    )
)

##### 📉 Visualization

In [25]:

rv_data_animated = (
    df
    .groupby(['Product', 'Month'], as_index=False)
    .agg(
        total_revenue=('Amount', 'sum'),
        total_boxes=('Boxes Shipped', 'sum')
    )
    .sort_values('Month')
)

# 2. Create Animated Scatter Plot
fig = px.scatter(
    rv_data_animated,
    x='total_boxes',
    y='total_revenue',
    animation_frame='Month',    
    animation_group='Product',  
    size='total_revenue',
    color='Product',            
    hover_name='Product',
    template='plotly_dark',
    title='Animated Revenue vs Volume (Boxes Shipped)',
    labels={'total_boxes': 'Boxes Shipped', 'total_revenue': 'Revenue'},
    range_x=[0, rv_data_animated['total_boxes'].max() * 1.1], 
    range_y=[0, rv_data_animated['total_revenue'].max() * 1.1]
)

# 3. Add Reference Line (from 0,0 to Max)
max_val = max(rv_data_animated['total_boxes'].max(), rv_data_animated['total_revenue'].max())

fig.add_shape(
    type="line",
    x0=0, y0=0, 
    x1=rv_data_animated['total_boxes'].max(), 
    y1=rv_data_animated['total_revenue'].max(),
    line=dict(color="White", width=2, dash="dot"),
)

# Add a label for the reference line
fig.add_annotation(
    x=rv_data_animated['total_boxes'].max(),
    y=rv_data_animated['total_revenue'].max(),
    text="Efficiency Baseline",
    showarrow=False,
    yshift=10
)

# Make animation smooth
fig.layout.updatemenus[0].buttons[0].args[1]["frame"]["duration"] = 800

fig.show()

##### 🔹 Q19: Customer Strategy Insight

- Which country-product combinations drive maximum revenue?

📊 Graph: Heatmap

In [26]:
heatmap_data = (
    df
    .pivot_table(
        values='Amount',
        index='Country',
        columns='Product',
        aggfunc='sum',
        fill_value=0
    )
)

##### 📊 Visualization

In [27]:
fig = px.imshow(
    heatmap_data.head(5),
    text_auto=True,
    color_continuous_scale='tempo',
    title='Revenue Heatmap (Country vs Product)',
    template='plotly_dark'
)

fig.show()